# 🩺 Diabetes Q&A Pipeline — Demo & Giải thích

Notebook này giải thích từng bước pipeline **KG + RAG** để trả lời câu hỏi về bệnh tiểu đường.

## Ba chế độ truy xuất

| Chế độ | Nguồn dữ liệu | Phù hợp khi |
|---|---|---|
| **RAG** | Vector search trên văn bản (WHO PDF, MedlinePlus, Bộ Y tế) | Câu hỏi cần giải thích chi tiết |
| **Knowledge Graph** | Đồ thị Hetionet (thuốc, triệu chứng, gen) | Câu hỏi về quan hệ thực thể |
| **Hybrid** | Kết hợp cả hai | Câu hỏi phức tạp cần cả hai nguồn |

## Luồng dữ liệu
```
câu hỏi
  ├─► kg.py:         find_entities() → graph_facts()   ──┐
  └─► retriever.py:  retrieve()                         ──┤ hybrid_context()
                                                           └─► format_context() → build_prompt()
                                                                └─► Claude API → câu trả lời
```

## 0. Cài đặt môi trường

In [ ]:
import os, sys

# Đảm bảo import được từ thư mục này
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

# Dùng SBERT (nếu muốn TF-IDF nhanh hơn: đặt thành 'tfidf')
os.environ.setdefault('EMBED_BACKEND', 'sbert')

from config import PROC, DOCS

# Kiểm tra dữ liệu đã được chuẩn bị
chunks_path  = PROC / 'chunks.jsonl'
triples_path = PROC / 'diabetes_triples.csv'

print(f"chunks.jsonl  : {'✅' if chunks_path.exists()  else '❌ Chạy ingest.py trước'}")
print(f"triples.csv   : {'✅' if triples_path.exists() else '❌ Chạy build_kg.py trước'}")

if chunks_path.exists():
    import json
    chunks = [json.loads(l) for l in chunks_path.read_text(encoding='utf-8').splitlines()]
    by_src = {}
    for c in chunks:
        key = c['source'].split('–')[0].strip()
        by_src[key] = by_src.get(key, 0) + 1
    print(f"\nTổng chunks: {len(chunks)}")
    for src, n in by_src.items():
        print(f"  {n:4d}  {src}")

## 1. Tải mô hình và đồ thị tri thức

- **Retriever**: tải `all-MiniLM-L6-v2` (SBERT) và tạo vector index cho toàn bộ chunks  
  Embedding được cache vào file `.npy` — lần đầu mất ~30 giây, các lần sau < 1 giây.
- **Graph**: đọc `diabetes_triples.csv` vào NetworkX `MultiDiGraph`

In [ ]:
from retriever import Retriever
from kg import load_graph

print("Đang tải Retriever (SBERT + vector index)...")
R = Retriever()
print(f"  ✅ {len(R.chunks)} chunks đã được index")

print("\nĐang tải Knowledge Graph...")
G = load_graph()
print(f"  ✅ {G.number_of_nodes()} nút, {G.number_of_edges()} cạnh")

## 2. Chế độ RAG — Vector Search

**Cách hoạt động:**
1. Mã hóa câu hỏi thành vector bằng SBERT
2. Tính cosine similarity với toàn bộ chunk vectors
3. Trả về top-K chunks có điểm cao nhất

> **Điểm cosine** nằm trong [0, 1]. Ngưỡng `MIN_SCORE = 0.30` — dưới ngưỡng → hệ thống từ chối trả lời.

In [ ]:
from config import MIN_SCORE, TOP_K

question = "What symptoms are associated with type 2 diabetes?"

docs = R.retrieve(question, top_k=TOP_K)

print(f"Câu hỏi : {question}")
print(f"MIN_SCORE: {MIN_SCORE}  |  TOP_K: {TOP_K}")
print(f"Điểm cao nhất: {docs[0]['score']:.3f} → {'ĐỦ bằng chứng' if docs[0]['score'] >= MIN_SCORE else 'KHÔNG đủ'}\n")

for i, d in enumerate(docs, 1):
    loc = f"{d['source']}, trang {d['page']}" if d['page'] else d['source']
    print(f"--- Chunk {i}  |  score: {d['score']:.3f}  |  {loc}")
    print(d['text'][:300])
    print()

## 3. Chế độ Knowledge Graph

**Cách hoạt động:**
1. `find_entities()` — duyệt tên nút trong đồ thị, so khớp substring với câu hỏi
2. `relations_for()` — phát hiện từ khóa để lọc loại quan hệ (triệu chứng, thuốc, gen...)
3. `graph_facts()` — lấy 1-hop edges xung quanh entity, format thành câu tiếng Việt

**6 loại quan hệ Hetionet được dùng:**

| Mã | Nghĩa |
|---|---|
| `CtD` | Compound **treats** Disease |
| `CpD` | Compound **palliates** Disease |
| `DpS` | Disease **presents** Symptom |
| `DrD` | Disease **resembles** Disease |
| `DlA` | Disease **localizes** Anatomy |
| `DaG` | Disease **associates** Gene |

In [ ]:
from kg import find_entities, graph_facts, relations_for

question = "What drugs treat type 2 diabetes mellitus?"

entities = find_entities(G, question)
wanted   = relations_for(question)

print(f"Câu hỏi  : {question}")
print(f"Entities : {entities}")
print(f"Relations: {wanted}\n")

for entity in entities:
    facts = graph_facts(G, entity, relations=wanted)
    print(f"=== Graph facts cho '{entity}' ===")
    for f in facts:
        print(f"  {f}")
    print()

## 4. Chế độ Hybrid — Kết hợp

`hybrid_context()` gọi cả KG và RAG, trả về dict với:
- `graph`: danh sách graph facts
- `docs`: danh sách chunks với cosine score
- `enough`: `True` nếu có graph facts **hoặc** best cosine ≥ MIN_SCORE
- `entities`: thực thể tìm được

Flag `enough` cho phép từ chối trả lời câu hỏi ngoài phạm vi.

In [ ]:
import hybrid as _hybrid_mod
from hybrid import hybrid_context, format_context

# Inject instances đã tải để tránh load lại
_hybrid_mod._G = G
_hybrid_mod._R = R

question = "What drugs treat type 2 diabetes mellitus?"
ctx = hybrid_context(question)

print(f"Entities    : {ctx['entities']}")
print(f"Graph facts : {len(ctx['graph'])} dòng")
print(f"Docs        : {len(ctx['docs'])} chunks  |  best score: {ctx['docs'][0]['score']:.3f}")
print(f"Enough      : {ctx['enough']}")
print()
print("=== Context đã format ===")
print(format_context(ctx)[:800], "...")

## 5. Xây dựng Prompt cho LLM

`build_prompt()` gói context thành prompt có cấu trúc:
- Hướng dẫn hệ thống: trả lời từ context, không bịa, trích dẫn nguồn
- Context (GRAPH FACTS + DOCUMENT EVIDENCE)
- Câu hỏi

In [ ]:
from hybrid import build_prompt

context_str = format_context(ctx)
prompt      = build_prompt(question, context_str)

print(prompt)

## 6. Gọi Claude API

Cần đặt biến môi trường `ANTHROPIC_API_KEY` trước khi chạy cell này.

In [ ]:
import anthropic

if not os.environ.get('ANTHROPIC_API_KEY'):
    print("⛔ Chưa đặt ANTHROPIC_API_KEY.")
    print("   Chạy: import os; os.environ['ANTHROPIC_API_KEY'] = 'sk-ant-...'")
else:
    client = anthropic.Anthropic()
    msg = client.messages.create(
        model="claude-3-5-haiku-20241022",
        max_tokens=600,
        messages=[{"role": "user", "content": prompt}],
    )
    answer = msg.content[0].text
    print(f"Câu hỏi: {question}\n")
    print("=" * 60)
    print(answer)

## 7. So sánh 3 chế độ với cùng một câu hỏi

Thay `QUESTION` bên dưới để thử với câu hỏi của bạn.

In [ ]:
QUESTION = "What symptoms are associated with type 2 diabetes mellitus?"

# --- RAG context ---
rag_docs = R.retrieve(QUESTION, top_k=TOP_K)
rag_ctx  = {"question": QUESTION, "entities": [], "graph": [],
            "docs": rag_docs, "enough": rag_docs[0]['score'] >= MIN_SCORE}

# --- KG context ---
ents    = find_entities(G, QUESTION)
wanted  = relations_for(QUESTION)
lines   = [f for e in ents for f in graph_facts(G, e, relations=wanted)]
kg_ctx  = {"question": QUESTION, "entities": ents,
           "graph": list(dict.fromkeys(lines)), "docs": [], "enough": bool(lines)}

# --- Hybrid context ---
hyb_ctx = hybrid_context(QUESTION)

print(f"Câu hỏi: {QUESTION}\n")
print(f"{'Chế độ':<12} {'Entities':>8} {'Graph facts':>12} {'Chunks':>8} {'Best score':>11} {'Enough':>8}")
print("-" * 65)
for name, ctx in [("RAG", rag_ctx), ("KG", kg_ctx), ("Hybrid", hyb_ctx)]:
    best = ctx['docs'][0]['score'] if ctx['docs'] else 0.0
    print(f"{name:<12} {len(ctx['entities']):>8} {len(ctx['graph']):>12} "
          f"{len(ctx['docs']):>8} {best:>11.3f} {'✅' if ctx['enough'] else '❌':>8}")

## 8. Câu hỏi ngoài phạm vi — Kiểm tra từ chối

Hệ thống **không nên** trả lời câu hỏi không liên quan đến tiểu đường.

In [ ]:
out_of_scope = [
    "Who won the 2022 FIFA World Cup?",
    "What is the recommended insulin dose for a 12-year-old child?",
]

for q in out_of_scope:
    ctx  = hybrid_context(q)
    best = ctx['docs'][0]['score'] if ctx['docs'] else 0.0
    verdict = "✅ Từ chối đúng" if not ctx['enough'] else "⚠️ Có thể trả lời"
    print(f"{verdict} | score={best:.3f} | {q}")

## Tóm tắt cho hội đồng

| Thành phần | Giải pháp | Lý do |
|---|---|---|
| Chunking | Sliding-window 150 từ, overlap 30 từ | SBERT giới hạn 256 token; overlap tránh mất thông tin |
| Embedding | `all-MiniLM-L6-v2` (SBERT) | Nhỏ gọn (90MB), tốc độ cao, chất lượng tốt cho câu ngắn |
| Knowledge Graph | Hetionet (lọc 6 metaedge liên quan tiểu đường) | Dữ liệu y sinh học có cấu trúc, quan hệ chính xác |
| Từ chối trả lời | Flag `enough` | Tránh hallucination khi không có bằng chứng |
| Nguồn tiếng Việt | Bộ Y tế QĐ 5481/2020 | Phù hợp ngữ cảnh lâm sàng Việt Nam |